210. Course Schedule II
Medium

There are a total of numCourses courses you have to take, labeled from 0 to numCourses - 1. You are given an array prerequisites where prerequisites[i] = [ai, bi] indicates that you must take course bi first if you want to take course ai.

    For example, the pair [0, 1], indicates that to take course 0 you have to first take course 1.

Return the ordering of courses you should take to finish all courses. If there are many valid answers, return any of them. If it is impossible to finish all courses, return an empty array.

Example 1:

Input: numCourses = 2, prerequisites = [[1,0]]
Output: [0,1]
Explanation: There are a total of 2 courses to take. To take course 1 you should have finished course 0. So the correct course order is [0,1].

Example 2:

Input: numCourses = 4, prerequisites = [[1,0],[2,0],[3,1],[3,2]]
Output: [0,2,1,3]
Explanation: There are a total of 4 courses to take. To take course 3 you should have finished both courses 1 and 2. Both courses 1 and 2 should be taken after you finished course 0.
So one correct course order is [0,1,2,3]. Another correct ordering is [0,2,1,3].

Example 3:

Input: numCourses = 1, prerequisites = []
Output: [0]

Constraints:

    1 <= numCourses <= 2000
    0 <= prerequisites.length <= numCourses * (numCourses - 1)
    prerequisites[i].length == 2
    0 <= ai, bi < numCourses
    ai != bi
    All the pairs [ai, bi] are distinct.


Hint 1
This problem is equivalent to finding the topological order in a directed graph. If a cycle exists, no topological ordering exists and therefore it will be impossible to take all courses.

Hint 2
Topological Sort via DFS - A great video tutorial (21 minutes) on Coursera explaining the basic concepts of Topological Sort.

Hint 3
Topological sort could also be done via BFS.

Analysis:

The problem asks for a valid order inwhich to finsh all courses. If prerequisites contain a cycle, return []

Each pair [course, prerequisites] means:
    Complete prerequisite before the course
So [1, 0] creates a directed edge 0 -> 1

Given data: 
numCourses = 4
courses = [0, 1, 2, 3]
prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]

The Pattern: 

We need an ordering where every prerequite appears before its dependent course. this is a toppological ordering of a directed graph 

- Vertices courses 0 through numCourses - 1
- Directed edges: prereqiusite -> dependent course 
- A cycle makes completion impossible 
- Disconnected courses must also appear in the answer 
- Multiple valid orders may may exist, return any one 

Naive Solution: try every possibel order 

Generate every permuation of the courses, For each permutation, check that every prerequisite appears before its dependent course. 

There are v! permutations. Building a position map and checking the edges takes O(V + E) per permutation 

So, Time O(V! * (V + E))

So, Optimal Approach will be Kahn's Algorithm using BFS 

What data structures we'll need 

| Data structure | Purpose |
|---|---|
| Adjacency list | For each course, store courses that depend on it |
| Indegree array | Number of prerequisites still outstanding for each course |
| Queue (`deque`) | Courses ready to take: indegree is `0` |
| Result list | Completed courses in order |

The central idea:
A course becomes available when its remaining prerequisite count reaches zero.

Steps:
1. Build the graph and indegree array.
2. Enqueue all courses with indegree 0.
3. Remove an available course and append it to the result.
4. Decrease the indegree of each course that depends on it.
5. If a dependent course reaches indegree 0, enqueue it.
6. Return the result if it contains every course; otherwise return [].


In [1]:
from collections import deque

def find_order(numCourses, prerequisites):
    graph = [[] for _ in range(numCourses)]

    indegree = [0] * numCourses

    # Prerequisite -> Course 
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    order = [] 

    # now include ever course that has no prerequisites 
    queue = deque(course for course in range(numCourses) if indegree[course] == 0)
    while queue:
        course = queue.popleft()
        order.append(course)

        # compelting this course satisfies one prerequisiite 
        # for each of its dependent courses 
        for dependent in graph[course]:
            indegree[dependent] -= 1 

            if indegree[dependent] == 0:
                queue.append(dependent)

    # Unprocessed courses indicate a cycle
    return order if len(order) == numCourses else [] 
    




# Given
numCourses = 4 
prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]
# Output: [0, 2, 1, 3]

result = find_order(numCourses, prerequisites)
print (result)


[0, 1, 2, 3]


In [2]:
def findOrderNaive(numCourses, prerequisites):
    order = []
    completed = set()
    
    # We need to pick numCourses elements. 
    # In the worst-case, we find exactly 1 course per full pass.
    for _ in range(numCourses):
        course_found_this_pass = False
        
        for course in range(numCourses):
            # If we've already taken this course, skip it
            if course in completed:
                continue
                
            # Naively scan the entire prerequisites list to see if 
            # this specific course has any incomplete prerequisites
            has_unmet_prereq = False
            for dest, src in prerequisites:
                if dest == course and src not in completed:
                    has_unmet_prereq = True
                    break # Can't take this course yet
            
            # If all prerequisites for this course are met, take it!
            if not has_unmet_prereq:
                order.append(course)
                completed.add(course)
                course_found_this_pass = True
                break # Break out to start a fresh pass with our new course
                
        # If we scanned every single course and couldn't complete a single one,
        # it means there's a circular dependency (cycle). Return an empty list.
        if not course_found_this_pass:
            return []
            
    return order

# Given
numCourses = 4 
prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]
# Output: [0, 2, 1, 3]

result = findOrderNaive(numCourses, prerequisites)
print (result)


[0, 1, 2, 3]



Why is this considered "Naive"? (Time & Space Complexity)

• Time Complexity: O(V^2 * E) (or O(numCourses² × prerequisites)).
	• The outer loop runs up to V times (once for each course we need to complete).
	• The inner loop checks all V courses to see which one is eligible.
	• Inside that check, it runs a nested linear search through all E dependency pairs to verify prerequisites.
	• Compared to the optimal O(V + E) topological sort, this will easily Time Limit Exceeded (TLE) on LeetCode for large inputs.
• Space Complexity: O(V). It is highly memory efficient since it only tracks the order array and the completed hash set, avoiding the memory overhead of constructing an adjacency list graph mapping.
